# <center>Pronóstico TimesFM 2.5 recursivo — ABCDE por ATRIBUTO (CCAA, Provincias, ...)</center>
## <center>Variante por desglose de forecast_ABCDE_estatal_TimesFM.ipynb</center>

> Reutiliza tal cual todas las piezas ya verificadas del notebook estatal (SSL/proxy, carga del
> modelo, funciones de fine-tuning, `recursive_forecast`, reglas por grupo ABC/DE). La diferencia
> real es que en vez de una sola serie, el CSV trae **varias columnas** (una por grupo del atributo:
> CCAA, provincia, sector...) y hay que repetir el pipeline completo (grid de `FT_CTX` + fine-tuning
> + pronóstico recursivo) **una vez por cada grupo** — igual que hace `forecast_ABC_atributo_NP.py`
> con su propio bucle.

**Diferencias respecto al notebook estatal:**
- **`VAL_MONTHS=12`** en vez de 36 — en modo atributo solo se reporta el MAPE del primer año, no
  de los tres (coincide con `NP_ABC_ATRIBUTO_PARAMS['val_months']`). Esto abarata la parte de
  *inferencia* del backtest (1 paso recursivo en vez de 3), pero **no** abarata el fine-tuning del
  backtest -- al quedar más histórico disponible (menos reservado para validación), salen *más*
  ventanas de entrenamiento. El entrenamiento FINAL (el que genera el pronóstico real) no depende
  de `VAL_MONTHS` en absoluto, porque siempre usa el histórico completo de cada grupo.
- **Bucle por grupo**: cada grupo hace su propio grid de `FT_CTX`, su propio fine-tuning y su
  propio pronóstico recursivo -- igual de riguroso que NP-atributo, pero mucho más caro por grupo
  que NP (aquí cada "combinación" implica fine-tuning de un modelo de 200M parámetros, no un ajuste
  ligero). **Por eso hay un `MAX_GRUPOS_PROBAR` en la configuración** -- pruébalo primero con pocos
  grupos para hacerte una idea real del tiempo antes de lanzarlo con el lote completo (hasta 53,
  el mismo límite que ya usa NP-atributo).
- `LOG_TRANSFORM` / `FIXED_LR` / `POINT_CHANNEL` siguen las mismas reglas por grupo de métrica
  (ABC vs DE) que en estatal -- no se re-optimizan por atributo, se heredan.

**Sin validar todavía**: esta metodología (bucle por grupo con grid de `FT_CTX` propio) no se ha
probado empíricamente como sí se hizo con Parados/Afiliados/Contratos en modo estatal. Los números
que salgan aquí son el primer contacto real con el coste y la calidad en modo atributo.

In [ ]:
# pip install "timesfm[torch]"   # ejecutar una sola vez en el entorno

import os
os.environ['USE_TF']    = '0'
os.environ['USE_TORCH'] = '1'

import ssl
ssl._create_default_https_context = ssl._create_unverified_context  # proxy corporativo Netskope (rutas stdlib / requests)

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import requests
_orig_request = requests.Session.request
def _patched_request(self, *a, **kw):
    kw.setdefault('verify', False)
    return _orig_request(self, *a, **kw)
requests.Session.request = _patched_request  # huggingface_hub < 1.0 usa requests por debajo

# huggingface_hub >= 1.0 usa httpx en vez de requests -- el parche de arriba no le afecta.
try:
    import httpx

    def _unverified_httpx_client_factory():
        return httpx.Client(verify=False, follow_redirects=True, timeout=None)

    import huggingface_hub
    huggingface_hub.set_client_factory(_unverified_httpx_client_factory)
    print('huggingface_hub (backend httpx) configurado sin verificación SSL -- proxy Netskope')
except (ImportError, AttributeError):
    pass  # huggingface_hub < 1.0 (sin httpx o sin set_client_factory) -- ya cubierto por el parche de requests de arriba

import copy
import json
import warnings
import datetime as _dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_percentage_error

import timesfm
from timesfm.torch import util as tfm_util

warnings.filterwarnings('ignore')

print('Imports OK | torch:', torch.__version__, '| timesfm:', timesfm.__package__ and getattr(timesfm, '__version__', 'n/d'))


## 1. Configuración

In [ ]:
# METRICA -- igual que en el notebook estatal
metrica = 'Parados'

# ATRIBUTO -- por que se desglosa: 'CCAA', 'Provincias', 'Sectores', 'Edad', ...
atributo = 'CCAA'

# Nombre de fichero segun la convencion del proyecto (ver CLAUDE.md):
# "{Metrica} desde {AnoInicio} por {Atributo}.csv"
CSV_PATH = f'{metrica} desde 2012 por {atributo}.csv'

TIMESFM_REPO_ID = 'google/timesfm-2.5-200m-pytorch'

# --- EXCEPCIONES MANUALES POR METRICA (lr y point_channel, igual que estatal) --
METRICAS_DE = ('Contratos', 'P. Contratadas')
FIXED_LR      = 5e-5 if metrica in METRICAS_DE else 5e-6
FIXED_LAYERS  = 4
POINT_CHANNEL = 0 if metrica == 'Contratos' else 5

# --- VALIDACION: 12 meses, no 36 (ver nota de la introduccion) -------------
VAL_MONTHS = 12

# --- FINE-TUNING (igual que estatal) ---------------------------------------
FT_CTX_GRID = [24, 36]
# LOG_TRANSFORM ya NO se decide por el nombre de la metrica (a diferencia del
# estatal): en modo atributo cada grupo puede tener una escala/estacionalidad
# muy distinta al agregado nacional (ver diagnostico de "Industria" en
# Parados por sector: sin log ~19-20% de MAPE, con log ~11%), asi que se
# incluye como una dimension mas del grid, junto con FT_CTX -- 2x2=4
# combinaciones por grupo. lr y point_channel se quedan fijos por regla
# ABC/DE (se probo que solos no ayudan y combinados no generalizan bien).
LOG_TRANSFORM_GRID = [False, True]
FT_HOR    = 12
FT_STEP   = 3
FT_EPOCHS = 15
RECURSIVE_STEP = 12

SEED = 11

# --- LIMITE DE GRUPOS PARA PROBAR --------------------------------------------
# None = todos los grupos del CSV. Pon un numero pequeno (3-5) para tu primera
# ejecucion -- cada grupo hace su propio grid de FT_CTX x LOG_TRANSFORM (4
# combinaciones) + fine-tuning final, asi que el coste total escala
# directamente con esto.
MAX_GRUPOS_PROBAR = 3

print(f'Metrica: {metrica}  |  Atributo: {atributo}  (grupo {"DE" if metrica in METRICAS_DE else "ABC"})')
print(f'FIXED_LR={FIXED_LR}  POINT_CHANNEL={POINT_CHANNEL}')
print(f'VAL_MONTHS={VAL_MONTHS}  FT_CTX_GRID={FT_CTX_GRID}  LOG_TRANSFORM_GRID={LOG_TRANSFORM_GRID}  MAX_GRUPOS_PROBAR={MAX_GRUPOS_PROBAR}')

## 2. Carga de datos y preparación

In [ ]:
# Misma convención que forecast_ABC_atributo_NP.py:
# encoding='latin1' por los caracteres especiales en nombres de provincias/CCAA;
# na_values=["'-"] convierte el marcador SEPE de dato confidencial en NaN.
df = pd.read_csv(CSV_PATH, sep=';', encoding='latin1', na_values=["'-"])
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df = df.sort_values('Fecha').reset_index(drop=True)

grupos_todos = list(df.columns[1:])   # todas las columnas menos 'Fecha'
grupos = grupos_todos[:MAX_GRUPOS_PROBAR] if MAX_GRUPOS_PROBAR is not None else grupos_todos

for g in grupos:
    df[g] = pd.to_numeric(df[g], errors='coerce')

n = len(df)
UltimaFechaHistorico = df['Fecha'].iloc[-1]

now = _dt.datetime.now().year
f_end = f'{now + 3}-12'
FECHA_FIN_PRONOSTICO = pd.Timestamp(f_end + '-01')
HORIZONTE_MESES = (
    (FECHA_FIN_PRONOSTICO.year  - UltimaFechaHistorico.year)  * 12 +
    (FECHA_FIN_PRONOSTICO.month - UltimaFechaHistorico.month)
)

print(f'Total: {n} meses | Grupos disponibles ({atributo}): {len(grupos_todos)} | Procesando: {len(grupos)}')
print(f'Grupos a procesar: {grupos}')
print(f"Último dato: {UltimaFechaHistorico.strftime('%Y-%m')}. Horizonte de pronóstico: {HORIZONTE_MESES} meses (hasta {f_end}).")

if n < FT_CTX_GRID[0] + VAL_MONTHS:
    raise ValueError(f'Histórico insuficiente ({n}m) para contexto={FT_CTX_GRID[0]}m + validación={VAL_MONTHS}m')


## 3. TimesFM 2.5 — descarga y carga del modelo base

Igual que en el notebook estatal -- se carga una sola vez y se reutiliza para el fine-tuning de
todos los grupos.

In [ ]:
# Ruta local del checkpoint. Ajusta si lo guardas en otro sitio.
TIMESFM_LOCAL_DIR = r'C:\Users\sgei044\Desktop\ML and IA with Python\Parados Contratos Afiliados 2026-2028\Modelos\TimesFM\timesfm-2.5-200m-pytorch'

if os.path.exists(os.path.join(TIMESFM_LOCAL_DIR, 'model.safetensors')):
    print(f'Cargando checkpoint local desde: {TIMESFM_LOCAL_DIR}')
    model = timesfm.TimesFM_2p5_200M_torch(torch_compile=False)
    model.load_checkpoint(TIMESFM_LOCAL_DIR)
else:
    print('No se encontró checkpoint local -- intentando descarga automática desde Hugging Face...')
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_REPO_ID, torch_compile=False)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=128,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=False,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

if HORIZONTE_MESES > model.forecast_config.max_horizon:
    raise ValueError(
        f'HORIZONTE_MESES ({HORIZONTE_MESES}) supera max_horizon '
        f'({model.forecast_config.max_horizon}) configurado en compile(). Aumenta max_horizon.'
    )

base_module = copy.deepcopy(model.model)

TFM_P          = model.model.p
TFM_O          = model.model.o
TFM_Q          = model.model.q
TFM_ARIDX      = model.model.aridx
TFM_NUM_LAYERS = len(model.model.stacked_xf)

Q_LOW, Q_MED, Q_HIGH = 1, TFM_ARIDX, 9    # p10, p50 (mediana=punto), p90

print(f'Modelo cargado — parámetros: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'p={TFM_P}  o={TFM_O}  q={TFM_Q}  aridx={TFM_ARIDX}  num_layers={TFM_NUM_LAYERS}')


## 4. Funciones auxiliares

Idénticas al notebook estatal (`forecast_ABCDE_estatal_TimesFM.ipynb`), ya verificadas contra el
código fuente real de `timesfm==3.0.0` -- ver ese notebook para el detalle de la verificación.

In [ ]:
def timesfm_forward_point(core_module, context_batch, mask_batch, horizon):
    """Pronóstico puntual diferenciable para fine-tuning.

    context_batch: FloatTensor (B, ctx_len); ctx_len debe ser múltiplo de core_module.p.
    mask_batch: BoolTensor (B, ctx_len); True donde context_batch es relleno (no datos reales).
    horizon: int <= core_module.o (128).
    Devuelve: FloatTensor (B, horizon).
    """
    p, o = core_module.p, core_module.o
    B, ctx_len = context_batch.shape
    assert ctx_len % p == 0, f'ctx_len ({ctx_len}) debe ser múltiplo de {p}'
    assert horizon <= o, f'horizon ({horizon}) debe ser <= {o}'

    patched_inputs = context_batch.reshape(B, -1, p)
    patched_masks  = mask_batch.reshape(B, -1, p)

    n_pts = torch.zeros(B, device=context_batch.device)
    mu    = torch.zeros(B, device=context_batch.device)
    sigma = torch.zeros(B, device=context_batch.device)
    patch_mu, patch_sigma = [], []
    for i in range(patched_inputs.shape[1]):
        (n_pts, mu, sigma), _ = tfm_util.update_running_stats(
            n_pts, mu, sigma, patched_inputs[:, i], patched_masks[:, i]
        )
        patch_mu.append(mu)
        patch_sigma.append(sigma)
    context_mu    = torch.stack(patch_mu, dim=1)
    context_sigma = torch.stack(patch_sigma, dim=1)

    normed_inputs = tfm_util.revin(patched_inputs, context_mu, context_sigma, reverse=False)
    normed_inputs = torch.where(patched_masks, 0.0, normed_inputs)

    (_, _, normed_outputs, _), _ = core_module(normed_inputs, patched_masks, decode_caches=None)

    renormed_outputs = tfm_util.revin(normed_outputs, context_mu, context_sigma, reverse=True)
    renormed_outputs = renormed_outputs.reshape(B, -1, o, core_module.q)

    return renormed_outputs[:, -1, :horizon, POINT_CHANNEL]


def pad_context_to_patch(ctx_arr, patch_len=32):
    """Rellena ctx_arr con ceros al PRINCIPIO hasta el siguiente múltiplo de
    patch_len (los datos reales quedan siempre al final). Devuelve (ctx_padded,
    mask) -- mask=True marca las posiciones de relleno."""
    L = len(ctx_arr)
    padded_len = ((L + patch_len - 1) // patch_len) * patch_len
    pad_amount = padded_len - L
    ctx_padded = np.concatenate([np.zeros(pad_amount, dtype=ctx_arr.dtype), ctx_arr])
    mask = np.concatenate([np.ones(pad_amount, dtype=bool), np.zeros(L, dtype=bool)])
    return ctx_padded, mask


def build_ft_windows(series, ctx, hor, step, patch_len=32):
    """Ternas (contexto rellenado, máscara, objetivo) por ventana deslizante."""
    windows = []
    for i in range(ctx, len(series) - hor + 1, step):
        ctx_arr, mask = pad_context_to_patch(series[i - ctx: i], patch_len)
        tgt_arr = series[i: i + hor]
        windows.append((ctx_arr, mask, tgt_arr))
    return windows


def make_ft_module(n_layers, base=None):
    """Copia profunda de `base` con solo las últimas `n_layers` capas de transformer
    + la cabeza de proyección puntual descongeladas."""
    base = base if base is not None else base_module
    ft_m = copy.deepcopy(base)
    for param in ft_m.parameters():
        param.requires_grad = False
    total = len(ft_m.stacked_xf)
    for i in range(total - n_layers, total):
        for param in ft_m.stacked_xf[i].parameters():
            param.requires_grad = True
    for param in ft_m.output_projection_point.parameters():
        param.requires_grad = True
    return ft_m


def run_ft_training(ft_m, lr, epochs, windows, seed=SEED, verbose=False):
    """Fine-tuning con AdamW, épocas FIJAS (sin early stopping)."""
    trainable_p = [p for p in ft_m.parameters() if p.requires_grad]
    optimizer   = torch.optim.AdamW(trainable_p, lr=lr)
    torch.manual_seed(seed)
    ft_m.train()
    for epoch in range(1, epochs + 1):
        epoch_losses = []
        for ctx_arr, mask_arr, tgt_arr in windows:
            x = torch.tensor(ctx_arr, dtype=torch.float32).unsqueeze(0)
            m = torch.tensor(mask_arr, dtype=torch.bool).unsqueeze(0)
            y = torch.tensor(tgt_arr, dtype=torch.float32).unsqueeze(0)
            pred = timesfm_forward_point(ft_m, x, m, horizon=len(tgt_arr))
            loss = F.mse_loss(pred, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_p, 1.0)
            optimizer.step()
            epoch_losses.append(loss.item())
        if verbose:
            print(f'    Época {epoch}/{epochs}: loss medio = {np.mean(epoch_losses):.6f}')
    ft_m.eval()
    return ft_m


def recursive_forecast(compiled_model, module, context, total_horizon, step=RECURSIVE_STEP):
    """Pronóstico de `total_horizon` meses encadenando pasos de `step` meses.
    Devuelve: (point_forecast, quantile_forecast), ambos en escala del modelo."""
    compiled_model.model = module
    ctx = list(context)
    points, quants = [], []
    remaining = total_horizon
    while remaining > 0:
        h = min(step, remaining)
        _, quant_block = compiled_model.forecast(horizon=h, inputs=[np.array(ctx)])
        quant_block = quant_block[0]
        point_block = quant_block[:, POINT_CHANNEL]
        points.append(point_block)
        quants.append(quant_block)
        ctx = ctx + list(point_block)
        remaining -= h
    return np.concatenate(points), np.concatenate(quants, axis=0)




print('Funciones auxiliares definidas.')


## 5. Bucle principal: pipeline completo por grupo

Para cada grupo: Zero-Shot de referencia, grid de `FT_CTX` sobre el backtest (`VAL_MONTHS=12`),
fine-tuning FINAL sobre el 100% del histórico de ese grupo, y pronóstico recursivo. Es el mismo
pipeline exacto del notebook estatal, repetido una vez por grupo -- por eso el aviso de coste de
la introducción.

In [ ]:
import itertools

resultados_grupo = {}
resumen_grupos   = []

n_grupos = len(grupos)
for gi, grupo in enumerate(grupos):
    print(f'\n{"="*70}\nGrupo {gi + 1}/{n_grupos}: {grupo}\n{"="*70}')

    all_values_g = df[grupo].values.astype(np.float32)
    n_g = len(all_values_g)

    if n_g < FT_CTX_GRID[0] + VAL_MONTHS or np.isnan(all_values_g).all():
        print(f'  [AVISO] Historico insuficiente o vacio para "{grupo}" -- se omite.')
        continue

    backtest_actual_g = all_values_g[n_g - VAL_MONTHS:]

    # Zero-Shot (referencia, siempre en escala real, sin log -- no participa en la eleccion de la combinacion)
    zs_bt_point_model, _ = recursive_forecast(model, base_module, all_values_g[:n_g - VAL_MONTHS], VAL_MONTHS)
    mape_zero_shot_g = mean_absolute_percentage_error(backtest_actual_g, zs_bt_point_model) * 100

    # Grid de (FT_CTX, LOG_TRANSFORM) sobre el backtest de este grupo -- 4 combinaciones
    best_mape_g = float('inf')
    ft_ctx_g = FT_CTX_GRID[0]
    log_transform_g = LOG_TRANSFORM_GRID[0]

    for candidate_ctx, candidate_log in itertools.product(FT_CTX_GRID, LOG_TRANSFORM_GRID):
        all_values_model_g = np.log(all_values_g) if candidate_log else all_values_g
        backtest_context_g = all_values_model_g[:n_g - VAL_MONTHS]

        ft_windows_bt = build_ft_windows(all_values_model_g[:-VAL_MONTHS], candidate_ctx, FT_HOR, FT_STEP)
        ft_m_bt = make_ft_module(FIXED_LAYERS)
        ft_m_bt = run_ft_training(ft_m_bt, FIXED_LR, FT_EPOCHS, ft_windows_bt, verbose=True)

        bt_point_model, _ = recursive_forecast(model, ft_m_bt, backtest_context_g, VAL_MONTHS)
        bt_point = np.exp(bt_point_model) if candidate_log else bt_point_model
        mape_combo = mean_absolute_percentage_error(backtest_actual_g, bt_point) * 100
        print(f'  FT_CTX={candidate_ctx}, LOG_TRANSFORM={candidate_log}: MAPE={mape_combo:.2f}%')

        if mape_combo < best_mape_g:
            best_mape_g = mape_combo
            ft_ctx_g = candidate_ctx
            log_transform_g = candidate_log
        del ft_m_bt

    print(f'  Mejor combinacion: FT_CTX={ft_ctx_g}, LOG_TRANSFORM={log_transform_g} (MAPE {best_mape_g:.2f}%)  |  Zero-Shot: {mape_zero_shot_g:.2f}%')

    # Fine-tuning FINAL sobre el 100% del historico de este grupo, con la mejor combinacion
    all_values_model_g = np.log(all_values_g) if log_transform_g else all_values_g
    ft_windows_final = build_ft_windows(all_values_model_g, ft_ctx_g, FT_HOR, FT_STEP)
    ft_model_final_g = make_ft_module(FIXED_LAYERS)
    ft_model_final_g = run_ft_training(ft_model_final_g, FIXED_LR, FT_EPOCHS, ft_windows_final, verbose=True)

    # Pronostico final recursivo (fine-tuned y zero-shot)
    forecast_dates_g = pd.date_range(
        UltimaFechaHistorico + pd.DateOffset(months=1), periods=HORIZONTE_MESES, freq='MS'
    )
    fc_point_model, fc_quant_model = recursive_forecast(model, ft_model_final_g, all_values_model_g, HORIZONTE_MESES)
    fc_point = np.exp(fc_point_model) if log_transform_g else fc_point_model
    fc_quant = np.exp(fc_quant_model) if log_transform_g else fc_quant_model

    zs_fc_point, zs_fc_quant = recursive_forecast(model, base_module, all_values_g, HORIZONTE_MESES)

    historico_g = [
        {'fecha': row['Fecha'].strftime('%Y-%m'), 'valor': round(float(row[grupo])) if pd.notna(row[grupo]) else None}
        for _, row in df.iterrows()
    ]
    pronostico_g = [{'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))} for d, v in zip(forecast_dates_g, fc_point)]
    intervalo_g = {
        'superior': [{'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))} for d, v in zip(forecast_dates_g, fc_quant[:, Q_HIGH])],
        'inferior': [{'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))} for d, v in zip(forecast_dates_g, fc_quant[:, Q_LOW])],
    }

    resultados_grupo[grupo] = {
        'historico': historico_g, 'pronostico': pronostico_g, 'intervalo_confianza': intervalo_g,
        'forecast_dates': forecast_dates_g, 'fc_point': fc_point, 'fc_quant': fc_quant,
        'zs_fc_point': zs_fc_point, 'zs_fc_quant': zs_fc_quant,
        'mape_fine_tuned': best_mape_g, 'mape_zero_shot': mape_zero_shot_g,
        'ft_ctx': ft_ctx_g, 'log_transform': log_transform_g,
    }
    resumen_grupos.append({
        'grupo': grupo, 'ft_ctx': ft_ctx_g, 'log_transform': log_transform_g,
        'mape_ft': round(best_mape_g, 2), 'mape_zs': round(mape_zero_shot_g, 2),
    })
    del ft_model_final_g

df_resumen = pd.DataFrame(resumen_grupos)
display(df_resumen)
print(f'\nMAPE medio (fine-tuned): {df_resumen["mape_ft"].mean():.2f}%')
print(f'MAPE medio (zero-shot):  {df_resumen["mape_zs"].mean():.2f}%')

## 6. Resultado (formato del proyecto) y exportación a Excel

In [ ]:
series_out = {
    g: {'historico': r['historico'], 'pronostico': r['pronostico'], 'intervalo_confianza': r['intervalo_confianza']}
    for g, r in resultados_grupo.items()
}

result = {
    'metrica':     metrica,
    'modo':        'atributo',
    'modelo':      'TimesFM',
    'atributo':    atributo,
    'anio_inicio': int(df['Fecha'].iloc[0].year),
    'series':      series_out,
    'mape':        round(float(df_resumen['mape_ft'].mean()), 2),
    # Extra fuera del contrato del proyecto -- solo para inspección en el notebook.
    'hiperparametros_por_grupo': {g: {'ft_ctx': r['ft_ctx'], 'log_transform': r['log_transform']} for g, r in resultados_grupo.items()},
}

out_file = f'MAPE {metrica} por {atributo} {f_end} TimesFM.xlsx'
with pd.ExcelWriter(out_file) as w:
    df_resumen.to_excel(w, sheet_name='Resumen', index=False)
    for g, r in resultados_grupo.items():
        df_fc_g = pd.DataFrame({
            'Fecha': r['forecast_dates'],
            f'{g}_FineTuned': r['fc_point'].round(0),
            'p10_FineTuned': r['fc_quant'][:, Q_LOW].round(0),
            'p90_FineTuned': r['fc_quant'][:, Q_HIGH].round(0),
            f'{g}_ZeroShot':  r['zs_fc_point'].round(0),
        })
        sheet_name = str(g)[:31]  # Excel limita el nombre de hoja a 31 caracteres
        df_fc_g.to_excel(w, sheet_name=sheet_name, index=False)
print(f'Exportado: {out_file}')


## 7. Visualizacion -- cuadricula de todos los grupos

Un panel por grupo (ultimos 36 meses de historico + pronostico), con el MAPE del
fine-tuned en el titulo -- mismo estilo que los cuadernos NP de atributo.

In [ ]:
import math

grupos_ok = list(resultados_grupo.keys())
n_plots = len(grupos_ok)
n_cols  = 5
n_rows  = math.ceil(n_plots / n_cols)

fig, axs = plt.subplots(n_rows, n_cols, figsize=(24, n_rows * 4))
axs = axs.flatten() if n_plots > 1 else [axs]

for i, grupo in enumerate(grupos_ok):
    r = resultados_grupo[grupo]

    # Ultimos 36 meses del historico
    hist_36 = df[['Fecha', grupo]].tail(36)
    axs[i].plot(hist_36['Fecha'], hist_36[grupo], color='blue', label='Real')

    # Pronostico fine-tuned + banda p10-p90
    axs[i].plot(r['forecast_dates'], r['fc_point'], color='red', label='Pronostico')
    axs[i].fill_between(r['forecast_dates'], r['fc_quant'][:, Q_LOW], r['fc_quant'][:, Q_HIGH],
                         color='red', alpha=0.15)

    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axs[i].tick_params(axis='x', rotation=45)
    titulo = grupo + '\nMAPE FT: {:.2f}% (ctx={}, log={})'.format(r['mape_fine_tuned'], r['ft_ctx'], r['log_transform'])
    axs[i].set_title(titulo, fontsize=12)
    axs[i].legend(fontsize=9)
    axs[i].grid(alpha=0.3)

for j in range(len(grupos_ok), len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()